In [0]:
# ===== CONFIGURACIÓN =====
from pyspark.sql import functions as F

SILVER = "workspace.easymoney_silver"
GOLD = "workspace.easymoney_gold"

def write_gold(df, table, comment):
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"{GOLD}.{table}"))
    spark.sql(f"COMMENT ON TABLE {GOLD}.{table} IS '{comment}'")

In [0]:
products = spark.table(f"{SILVER}.customer_products")
socio = spark.table(f"{SILVER}.customer_sociodemographics")
activity = spark.table(f"{SILVER}.customer_commercial_activity")

KEYS = ["customer_id", "partition_month"]
PRODUCT_COLS = [c for c in products.columns if c not in KEYS]

customer_monthly = (
    products
    .join(socio, KEYS, "inner")
    .join(activity, KEYS, "inner")
    .withColumn("n_products", sum(F.col(c) for c in PRODUCT_COLS))
    .withColumn("tenure_months",
                F.months_between("partition_month", "entry_month").cast("int"))
)

In [0]:
n_expected = products.count()
n_result = customer_monthly.count()
assert n_result == n_expected, (
    f"El join ha perdido {n_expected - n_result:,} filas: "
    "las tres tablas no tienen exactamente los mismos pares cliente-mes"
)
print(f"OK: {n_result:,} filas, {len(customer_monthly.columns)} columnas")

In [0]:
write_gold(customer_monthly, "customer_monthly",
    "Vista unica de cliente por mes. Grano: customer_id + partition_month. "
    "Une customer_products, customer_sociodemographics y customer_commercial_activity. "
    "n_products = numero de productos contratados; tenure_months = antiguedad en meses.")

In [0]:
kpi_customers = customer_monthly.groupBy("partition_month").agg(
    F.count("*").alias("customers"),
    F.sum("is_active").alias("active_customers"),
    F.round(F.avg("is_active") * 100, 1).alias("pct_active"),
    F.round(F.avg("n_products"), 2).alias("avg_products_per_customer"),
    *[F.round(F.avg(c) * 100, 2).alias(f"pct_{c}") for c in PRODUCT_COLS],
).orderBy("partition_month")

write_gold(kpi_customers, "kpi_customers_monthly",
    "KPIs mensuales de base de clientes: volumen, actividad y penetracion por producto (pct).")

display(kpi_customers)

In [0]:
kpi_sales = (
    spark.table(f"{SILVER}.sales")
    .join(spark.table(f"{SILVER}.product_description"), "product_id")
    .groupBy("sale_month", "product_family")
    .agg(
        F.count("*").alias("n_sales"),
        F.round(F.sum("net_margin"), 2).alias("net_margin_total"),
        F.round(F.avg("net_margin"), 2).alias("net_margin_avg"),
    )
    .orderBy("sale_month", "product_family")
)

write_gold(kpi_sales, "kpi_sales_monthly",
    "KPIs mensuales de ventas por familia de producto: numero de ventas y margen neto.")

display(kpi_sales)

In [0]:
from pyspark.sql import Window

TARGET = "pension_plan"                               # cambia aquí para otro producto
EXCLUDED_PRODUCTS = ["em_account_p", "em_account_pp"]  # penetración 0%
KEYS = ["customer_id", "partition_month"]

cm = spark.table(f"{GOLD}.customer_monthly")
PRODUCT_COLS = [c for c in spark.table(f"{SILVER}.customer_products").columns if c not in KEYS]
product_features = [c for c in PRODUCT_COLS if c not in EXCLUDED_PRODUCTS + [TARGET]]

# Fuente de verdad del target: ventas reales del producto en t+1, alineadas al mes t
target_product_ids = (spark.table(f"{SILVER}.product_description")
    .filter(F.col("product_name") == TARGET).select("product_id"))
sales_next = (spark.table(f"{SILVER}.sales")
    .join(target_product_ids, "product_id")
    .select("customer_id", F.add_months("sale_month", -1).alias("partition_month"))
    .distinct()
    .withColumn("has_sale_next_month", F.lit(1)))

w = Window.partitionBy("customer_id").orderBy("partition_month")
w_hist = w.rowsBetween(Window.unboundedPreceding, -1)   # todos los meses anteriores a t

base = (cm
    .withColumn("next_month", F.lead("partition_month").over(w))
    .withColumn("next_flag", F.lead(TARGET).over(w))
    .withColumn("prev_month", F.lag("partition_month").over(w))
    .withColumn("prev_n_products", F.lag("n_products").over(w))
    .withColumn("prev_is_active", F.lag("is_active").over(w))
    .withColumn("ever_had_before", F.coalesce(F.max(TARGET).over(w_hist), F.lit(0)))
    .join(sales_next, KEYS, "left")
)

has_next = F.col("next_month") == F.add_months("partition_month", 1)
has_prev = F.col("prev_month") == F.add_months("partition_month", -1)

eligible = base.filter(
    has_next
    & (F.col(TARGET) == 0)
    & (F.col("ever_had_before") == 0)     # nunca lo ha tenido: solo contrataciones nuevas
    & ~F.col("is_deceased")
    & (F.col("age") >= 18)
)

propensity = eligible.select(
    *KEYS,
    "age", "gender", "country_id", "region_code",
    "salary", F.col("salary").isNull().cast("int").alias("salary_missing"),
    "is_active",
    F.when(has_prev, F.col("prev_is_active")).alias("was_active_prev_month"),
    F.coalesce("segment_name", F.lit("UNKNOWN")).alias("segment"),
    F.coalesce("entry_channel", F.lit("UNKNOWN")).alias("entry_channel"),
    "tenure_months", "n_products",
    F.when(has_prev, F.col("n_products") - F.col("prev_n_products")).alias("n_products_delta_1m"),
    *product_features,
    F.coalesce("has_sale_next_month", F.lit(0)).alias("target"),
)

In [0]:
n = propensity.count()
n_keys = propensity.select(*KEYS).distinct().count()
n_bad_target = propensity.filter(~F.col("target").isin(0, 1) | F.col("target").isNull()).count()
last_month_cm = cm.agg(F.max("partition_month")).first()[0]
last_month_prop = propensity.agg(F.max("partition_month")).first()[0]

assert n == n_keys, f"{n - n_keys:,} duplicados cliente-mes"
assert n_bad_target == 0, f"{n_bad_target:,} targets fuera de 0/1"
assert TARGET not in propensity.columns, "FUGA: el target del mes t está entre las features"
assert last_month_prop < last_month_cm, "El último mes no tiene t+1 y no debería estar"

print(f"OK: {n:,} filas | último mes con target: {last_month_prop} | {len(propensity.columns)} columnas")

display(propensity.groupBy("partition_month").agg(
    F.count("*").alias("clientes_elegibles"),
    F.sum("target").alias("contrataciones"),
    F.round(F.avg("target") * 100, 3).alias("tasa_pct"),
).orderBy("partition_month"))
# Reconciliación entre fuentes: en la población elegible, venta y flag deberían coincidir
n_mismatch = eligible.filter(
    F.coalesce("has_sale_next_month", F.lit(0)) != F.col("next_flag")
).count()
print(f"Discrepancias venta vs flag en población elegible: {n_mismatch:,}")

In [0]:
write_gold(propensity, "propensity_features",
    "Features para propension de pension_plan. Grano: customer_id + partition_month (t). "
    "target = venta real de pension_plan en t+1 (fuente: silver.sales). "
    "Poblacion: nunca ha tenido el producto, mayor de edad, no fallecido. "
    "Las reactivaciones del flag (~50 por ciento de los 0 a 1) se excluyen: no generan venta. "
    "salary sin imputar (imputacion en entrenamiento) + flag salary_missing.")
